In [1]:
# 0. constants and raster paths

CELL_SIZE  = 5.0   # DEM resolution (m)
SLOPE_MEAN = 25.9  # deg, from slope raster stats
SLOPE_STD  = 9.5   # deg
MIN_ELEV   = 801.0 # m, for plotting later

# breach tolerance: cell may "breach" uphill up to this slope angle
BREACH_ANGLE = (SLOPE_MEAN - 2*SLOPE_STD) / 2
assert BREACH_ANGLE > 1e-5
print("breach angle:", BREACH_ANGLE)

# combined slope+basin DEM domains = routing DEM
dem_path    = "../data/rasters/canwell_domainlong_elev2000.tif"

# slope-domain-only feature rasters
slope_path  = "../data/rasters/canwell_nslopelong_slope2000.tif"
aspect_path = "../data/rasters/canwell_nslopelong_aspect2000.tif"
ndvi_path   = "../data/rasters/canwell_nslopelong_ndvi2000.tif"
lst_path    = "../data/rasters/canwell_nslopelong_lst2000.tif"
ddem_path   = "../data/rasters/canwell_nslopelong_ddem_2010_2000.tif"

# basin outline
basin_path  = "../data/outlines/ncanwell_longbasin_nolap.gpkg"
print("paths set")

breach angle: 3.4499999999999993
paths set


In [2]:
# 0.1 basin mask from outline polygon
import rasterio
import geopandas as gpd
from rasterio.features import geometry_mask

def make_basin_mask(dem_file, basin_file):
    """
    Rasterize a basin outline polygon onto the DEM grid.
    Returns: basin_mask (bool, True inside basin), transform, extent [L,R,B,T].
    """
    with rasterio.open(dem_file) as src:
        dem_crs   = src.crs
        transform = src.transform
        shape     = (src.height, src.width)
        b         = src.bounds
    extent = [b.left, b.right, b.bottom, b.top]

    basin_poly = gpd.read_file(basin_file).to_crs(dem_crs)
    basin_mask = geometry_mask(
        basin_poly.geometry,
        out_shape=shape,
        transform=transform,
        invert=True,
    )
    return basin_mask, transform, extent

In [3]:
# priority-flood + epsilon fill toward the basin outlet (routing DEM only)
# Barnes priority flood algorithm (Barnes, Lehman, Mulla, 2014)
import heapq
import numpy as np

def priority_flood_fill(dem, valid, basin_mask, eps=1e-3):
    H, W = dem.shape
    filled = dem.astype(np.float64).copy()
    done = np.zeros((H, W), bool)
    pq = []
    for r, c in zip(*np.where(basin_mask & valid)):   # basin = outlets
        done[r, c] = True
        heapq.heappush(pq, (filled[r, c], r, c))
    nb = [(-1,0),(1,0),(0,1),(0,-1),(-1,-1),(-1,1),(1,-1),(1,1)]
    while pq:
        e, r, c = heapq.heappop(pq)
        for dr, dc in nb:
            nr, nc = r+dr, c+dc
            if 0 <= nr < H and 0 <= nc < W and valid[nr, nc] and not done[nr, nc]:
                done[nr, nc] = True
                filled[nr, nc] = max(filled[nr, nc], e + eps)   # just-above downstream
                heapq.heappush(pq, (filled[nr, nc], nr, nc))
    reached = int((done & valid).sum()); total = int(valid.sum())
    print(f"flood reached {reached:,}/{total:,} valid cells")
    return filled

In [4]:
# 0.2 breached D8 directed forest over SLOPE cells only
# using the routing dem to trace paths, orig. dem for elev feature
# --- basin = mushgraph, added later

import numpy as np
import rasterio
import networkx as nx
from scipy.ndimage import distance_transform_edt
import psutil

def diGraph_from_dem(dem_path, slope_path, aspect_path, ndvi_path, lst_path, ddem_path,
                     basin_mask, cell_size=CELL_SIZE, breach_angle=BREACH_ANGLE,
                     basin_level=-99.0):
    """
    Downhill directed forest over the SLOPE cells of a merged slope+basin DEM. Basin cells
    are floored to basin_level so slope flow drains into the basin; basin cells are NOT made
    into nodes. A slope cell whose steepest descent exits into the basin is a 'pit' (tree root).

    Node attrs (slope only): dem_idx, elev, slope, aspect_sin, aspect_cos, ndvi, lst,
        ddem (target), dist_euclid (m to nearest basin cell), path_to_pit (-1 placeholder),
        is_pit, is_source, is_localSink, is_absSink.   Edge attr: weight = |normalized slope|.
    """

    with rasterio.open(dem_path) as src:
        dem = src.read(1); nodata = src.nodata; H, W = dem.shape

    def read_like(path, name):
        with rasterio.open(path) as s: a = s.read(1)
        if a.shape != (H, W): raise ValueError(f"{name} {a.shape} != DEM {(H,W)}")
        return a

    slope_data = read_like(slope_path, "slope")
    aspect     = read_like(aspect_path, "aspect")
    ndvi_data  = read_like(ndvi_path, "ndvi")
    lst_data   = read_like(lst_path, "lst")
    ddem_data  = read_like(ddem_path, "ddem")

    asp = np.deg2rad(aspect.astype(np.float64))
    asp_sin = np.sin(asp).astype(np.float32); asp_cos = np.cos(asp).astype(np.float32)

    # determine valid cells
    valid = dem != nodata                    # total real (slope or basin domain) cells
    dem_mod = dem.copy(); dem_mod[basin_mask] = basin_level # dem with basin pushed down
    dem_route = priority_flood_fill(dem_mod, valid, basin_mask) # build routing dem
    
    slope_cell = valid & (~basin_mask)       # slope domain nodes only
    dist_euclid = distance_transform_edt(~basin_mask).astype(np.float32) * cell_size

    graph = nx.DiGraph()
    sourceList, pitList, localSinkList, absSinkList = [], [], [], []
    neighbors = [(-1,0),(1,0),(0,1),(0,-1),(-1,-1),(-1,1),(1,-1),(1,1)]

    for r in range(H):
        if r % 100 == 0:
            print(f"Row {r}/{H}, RAM: {psutil.virtual_memory().percent:.1f}%")
        for c in range(W):
            if not slope_cell[r, c]:
                continue # skip nodata and basin cells
            idx = r * W + c # single count across raster
            graph.add_node(idx, dem_idx=[r, c],
                           elev=float(dem[r, c]), slope=float(slope_data[r, c]),
                           aspect_sin=float(asp_sin[r, c]), aspect_cos=float(asp_cos[r, c]),
                           ndvi=float(ndvi_data[r, c]), lst=float(lst_data[r, c]),
                           ddem=float(ddem_data[r, c]),           # target
                           dist_euclid=float(dist_euclid[r, c]),
                           path_dist_pit=-1,                         # filled after trees built
                           is_pit=False, is_source=False,
                           is_localSink=False, is_absSink=False)

            # for a given node:
            minSlope, maxSlope = breach_angle, 0.0
            next_node, next_is_basin = idx, False
            for dr, dc in neighbors:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < H and 0 <= nc < W): continue # valid check
                if not valid[nr, nc]: continue # never route into off-domain nodata

                s = (dem_route[nr, nc] - dem_route[r, c]) / (np.hypot(dr, dc) * cell_size)
                if s < minSlope: # steeper downslope->new next node; check for basin termination
                    minSlope = s; next_node = nr * W + nc; next_is_basin=bool(basin_mask[nr, nc])
                if s > maxSlope:
                    maxSlope = s

            # assign node type features
            if minSlope < breach_angle - 1e-5: # has a (possibly breached) outlet
                if next_is_basin:              # steepest descent exits to basin
                    graph.nodes[idx]["is_pit"] = True
                    pitList.append(idx)
                else:
                    graph.add_edge(idx, next_node, weight=abs(minSlope))
                    if minSlope < 0 and maxSlope == 0: # source: outflow, no inflow
                        graph.nodes[idx]["is_source"] = True
                        sourceList.append(idx)
                    elif minSlope >= 0:
                        graph.nodes[idx]["is_localSink"] = True
                        localSinkList.append(idx)
            else: # no (breached) outlet in any direction, no basin
                graph.nodes[idx]["is_absSink"] = True
                absSinkList.append(idx)
                
    print(f"nodes: {graph.number_of_nodes():,}  edges: {graph.number_of_edges():,}")
    print(f"sources {len(sourceList):,} | pits {len(pitList):,} | "
          f"breached sinks {len(localSinkList):,} | abs sinks {len(absSinkList):,}")                
                    
    return graph, sourceList, pitList, localSinkList, absSinkList, dem, dem_mod        

In [5]:
# build basin mask to use in graph construction
# transform and extent for later use
basin_mask, transform, extent = make_basin_mask(dem_path, basin_path)

In [6]:
# retrieve the directed dem graph and the lists of basin pit and other type nodes
sledGraph, sourceList, basinPitList, localSinkList, absSinkList, dem_data, dem_mod = diGraph_from_dem(
    dem_path, slope_path, aspect_path, ndvi_path, lst_path, ddem_path, basin_mask)

flood reached 1,723,466/1,723,466 valid cells
Row 0/1545, RAM: 13.7%
Row 100/1545, RAM: 13.2%
Row 200/1545, RAM: 13.6%
Row 300/1545, RAM: 14.1%
Row 400/1545, RAM: 14.7%
Row 500/1545, RAM: 15.7%
Row 600/1545, RAM: 16.6%
Row 700/1545, RAM: 17.8%
Row 800/1545, RAM: 18.8%
Row 900/1545, RAM: 19.9%
Row 1000/1545, RAM: 21.5%
Row 1100/1545, RAM: 22.3%
Row 1200/1545, RAM: 23.0%
Row 1300/1545, RAM: 23.3%
Row 1400/1545, RAM: 23.3%
Row 1500/1545, RAM: 23.3%
nodes: 887,825  edges: 884,105
sources 4,625 | pits 3,720 | breached sinks 0 | abs sinks 0


In [7]:
# 2.1 fill path_dist_pit for all nodes with multisource BFS
from collections import deque
def assign_path_dist_pit(graph, pit_list):
    dq = deque()
    for p in pit_list:
        graph.nodes[p]["path_dist_pit"] = 0
        dq.append(p)
    while dq:
        cur = dq.popleft(); d = graph.nodes[cur]["path_dist_pit"]
        for pred in graph.predecessors(cur):
            if graph.nodes[pred]["path_dist_pit"] == -1:
                graph.nodes[pred]["path_dist_pit"] = d + 1
                dq.append(pred)

    unreached = sum(1 for n in graph if graph.nodes[n]["path_dist_pit"] == -1)
    print(f"path_dist_pit filled | unreached nodes: {unreached:,}")
    return unreached

In [8]:
# assign path_dist_pit feature values
unreached = assign_path_dist_pit(sledGraph, basinPitList)

path_dist_pit filled | unreached nodes: 0


In [9]:
# checkpoint 1
save_graph = {
    "sledgraph_routebreach": sledGraph,    # slope-only directed forest, all node/edge attrs
    "dem_data":    dem_data,               # original elevation (plotting)
    "dem_mod":     dem_mod,                # basin floored to basin_level (routing reference)
    "basinPitList":     basinPitList,                # slope-side toe pits (tree roots)
    "sourceList":  sourceList,
    "localSinkList": localSinkList,
    "absSinkList":   absSinkList,          # empty this run, kept for schema consistency
    "basin_mask":  basin_mask,
    "transform":   transform,
    "extent":      extent,
}

In [10]:
import pickle
with open('../data/constructs/CGTN_sledgraph_routebreach.pkl','wb') as f:
    pickle.dump(save_graph,f)
print("saved!")

saved!


In [11]:
# 2.2 -- trace one tree upstream of a pit (for plotting),
#       with each node's edge-count (path) distance from pit (root)

def trace_tree_from_pit(graph, pit_node):
    """
    All nodes flowing into pit_node (BFS over predecessors).
    (BFS -- breadth-first search)
    Returns (tree_subgraph, depth), where depth[node] = edge count to the pit.
    Terminates at one out-edge per node; depth dic guards from cycles
    """

    depth = {pit_node: 0}
    order = [pit_node]; i = 0
    while i < len(order):
        cur = order[i]; i += 1
        for p in graph.predecessors(cur):
            if p not in depth:
                depth[p] = depth[cur] + 1
                order.append(p)
    return graph.subgraph(depth.keys()).copy(), depth

In [12]:
# 2.3 keep drainage trees larger than a threshold (50)
basinPit_trees = {}
threshold = 50
for pit_idx in basinPitList:
    tree, depth = trace_tree_from_pit(sledGraph, pit_idx)
    if tree.number_of_nodes() > threshold:
        basinPit_trees[pit_idx] = tree
print(f"basinPit trees size > {threshold}: {len(basinPit_trees)}")

basinPit trees size > 50: 286


In [14]:
threshold_trees = basinPit_trees.keys()
threshold_tree_sizes = [basinPit_trees[tree].number_of_nodes() for tree in threshold_trees]
tree_node_coverage = np.sum(threshold_tree_sizes)
print("threshold_tree_coverage:", tree_node_coverage, " nodes ", 
      "| ", (tree_node_coverage/sledGraph.number_of_nodes())*100, "%")

threshold_tree_coverage: 876690  nodes  |  98.74581139301101 %


In [23]:
# connect drainage trees to basin (mush)graph
import numpy as np
from scipy.spatial import cKDTree
import rasterio

# basin mushgraph
bg = np.load("../data/constructs/basin_mushgraph.npz", allow_pickle=True)

blevel, bx, by = bg["level"], bg["x"], bg["y"]
feat_names = list(bg["feat_names"])
belev = bg["X"][:, feat_names.index("elev")]
all_idx = np.arange(len(blevel)) # assign node indices for all basin nodes
L0, L2 = all_idx[blevel == 0], all_idx[blevel == 2] # subselect levels 0 and 2

# KDTree -- nearest-neighbor lookups in K-dimensional space
tree0 = cKDTree(np.c_[bx[L0], by[L0]]) # build lookup trees for the x,y coord
#tree2 = cKDTree(np.c_[bx[L2], by[L2]]) #      of nodes per level

pit_nodes = threshold_trees       # 286 pit roots (as sledGraph node ids)
tree_sizes = threshold_tree_sizes # node counts of the trees
DEM_FILE   = dem_path             # domain dem the full sledGraph was built on
P = len(pit_nodes)

with rasterio.open(DEM_FILE) as src:
    T = src.transform
    # W = src.width

def idx_xy(i):
    # r, c = divmod(int(i), W) # returns (quotient, remainder) <-> (row, column)
    return (T.c + (c + 0.5)*T.a, T.f + (r + 0.5)*T.e) # return coordinate x, y from raster

#pit_demidx = np.array([sledGraph.nodes[n]["dem_idx"] for n in pit_nodes]) # node attr
pit_rc = np.array([sledGraph.nodes[n]["dem_idx"] for n in pit_nodes])
pit_elev   = np.array([sledGraph.nodes[n]["elev"]    for n in pit_nodes])

#pit_xy     = np.array([idx_xy(i) for i in pit_demidx]) # coordinates for all pit nodes

rows, cols = pit_rc[:, 0], pit_rc[:, 1]
pit_xy = np.c_[T.c + (cols + 0.5) *T.a,   # x = origin_x + (col + 1/2)*pixel
                T.f + (rows + 0.5) * T.e]  # y = origin_y + (row + 1/2)*pixel(neg)

# L0: each pit connects to nearest L0 node
d0, j0 = tree0.query(pit_xy) # distances, indices <-- query
b0 = L0[j0] # level-0 basin nodes closest to the queried pit nodes

# L2: each pit connects to all L2 nodes
pit_L2 = np.repeat(np.arange(P), len(L2))
b2_all = np.tile(L2, P)

# closest-only L2 version
# d2, j2 = tree2.query(pit_xy)
# b2 = L2[j2] # level-2 basin nodes closest to the queried pit nodes

def edge_attr(pit_i, b):
    dx = bx[b] - pit_xy[pit_i,0] # distance in x, y coordinates
    dy = by[b] - pit_xy[pit_i,1]
    d = np.hypot(dx, dy)
    u = np.divide(1.0, d, out=np.zeros_like(d), where=(d>0)) # 1/d where d>0; 0 otherwise
    #u = np.where(d > 0, 1.0/d, 0.0) # throws divide-by-zero error, although dvz cells are ignored
    return np.c_[d, dx*u, dy*u, belev[b] - pit_elev[pit_i]] # 1D arrays into columns of 2D array

link_pit   = np.r_[np.arange(P), pit_L2]
link_basin = np.r_[b0,           b2_all]
link_kind  = np.r_[np.zeros(P, int), np.ones(len(pit_L2),int)]
link_attr  = np.vstack([edge_attr(np.arange(P), b0),
                        edge_attr(pit_L2,  b2_all)])

# closest-only L2 version
#link_pit   = np.r_[np.arange(P), np.arange(P)]
#link_basin = np.r_[b0, b2]
#link_kind  = np.r_[np.zeros(P, int), np.ones(P, int)] # 0=L0, local; 1=L2, global
#link_attr  = np.vstack([attr(b0), attr(b2)]) # directions and distances to linked nodes


In [26]:
np.savez("../data/constructs/pit_basin_links.npz",
         pit_dem_idx=pit_rc, pit_xy=pit_xy, pit_elev=pit_elev, tree_size=tree_sizes,
         link_pit=link_pit, link_basin=link_basin, link_kind=link_kind, link_attr=link_attr,
         attr_names=np.array(["dist_m","ux","uy","dx"]))

print(f"{P} pits wired | L0 nn: {d0.mean():.0f}±{d0.std():.0f} m (max {d0.max():.0f}) ")
      #f"| L2 nn: {d2.mean():.0f} m (max {d2.max():.0f})")

286 pits wired | L0 nn: 6±3 m (max 11) 


In [31]:
# save basin pit trees

import pickle

FEATS = ["elev", "slope", "aspect_sin", "aspect_cos",
         "ndvi", "lst", "dist_euclid", "path_dist_pit"]   # 8 inputs; ddem is the target
# excluded: dem_idx (coords), is_pit/is_source/is_localSink/is_absSink (flags)

basinPit_trees = {k: g.copy() for k, g in basinPit_trees.items()}   # make pickle-safe / self-contained

sled_handoff = {
    'basinPit_trees': basinPit_trees,   # 286 threshold trees, keyed by pit_idx
    'pit_nodes':      list(pit_nodes),  # ORDERED pit keys — must match pit_basin_links link_pit
    'FEATS':          FEATS,            # feature-name order for X assembly
    'transform':      transform,        # coord conversions
    'dem_data':       dem_data,         # plotting
    'extent':         extent,           # spatial plots
    'basin_mask':     basin_mask,       # overlay (the 'bluffs' analog)
    # 'sledGraph':    sledGraph,        # optional — only if you want the fringe nodes for plots
}

with open("../data/constructs/sledgraph_torch_treehandoff.pkl", "wb") as f:
    pickle.dump(sled_handoff, f, protocol=pickle.HIGHEST_PROTOCOL)
print("saved")

saved


In [33]:
# 3.0 flatten -- build slope tensors and pit-index glue

import numpy as np

node2idx = {}
rows_X, rows_y, src, dst = [], [], [], []

for pit_idx, tree in basinPit_trees.items():
    for n, a in tree.nodes(data=True):
        if n not in node2idx:
            node2idx[n] = len(node2idx) # iteratively new node index
            rows_X.append([a[f] for f in FEATS])
            rows_y.append(a["ddem"])
    for u, v in tree.edges():   # downslope directed (u -> v)
        src.append(node2idx[u]); dst.append(node2idx[v])

X_slope = np.asarray(rows_X, dtype=np.float32)
y_slope = np.asarray(rows_y, dtype=np.float32)
edge_slope = np.asarray([src, dst], dtype=np.float32)
pit_torch_idx = np.asarray([node2idx[n] for n in pit_nodes], dtype=np.int64) # link_pit -> slope_idx

np.savez("../data/constructs/slope_torch.npz",
         X=X_slope, y=y_slope, edge_index=edge_slope,
         feat_names=np.array(FEATS), pit_torch_idx=pit_torch_idx)
print(f"slope nodes {X_slope.shape[0]:,} | feats {X_slope.shape[1]} "
      f"| edges {edge_slope.shape[1]:,} | pits {len(pit_torch_idx)}")

slope nodes 876,690 | feats 8 | edges 876,404 | pits 286
